# 03 — Feature Engineering & Pairwise Metric Construction

**Transforming string pairs into a 15-dimensional feature vector**

Each benchmark pair is converted into a numerical vector of 15 similarity features capturing name, address, and structural signals. These features form the input matrix for all downstream classifiers. Good feature engineering is what separates a 60% model from a 95% model on entity resolution tasks.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from src.features import FEATURE_COLUMNS

sns.set_theme(style='whitegrid')
%matplotlib inline
print(f"FEATURE_COLUMNS ({len(FEATURE_COLUMNS)} features):", FEATURE_COLUMNS)

### 1. Load Feature Matrix

`data/processed/feature_matrix.csv` contains pre-computed features for every pair in the benchmark. Loading this avoids re-computing features during model training notebooks.

In [ ]:
feat_df = pd.read_csv('data/processed/feature_matrix.csv')
print(f'Shape: {feat_df.shape}')
print(f'Columns: {list(feat_df.columns)}')
feat_df.head(4)

### 2. The 15 Features — What They Measure

Below is the feature dictionary with the intuition behind each metric. The features are grouped into: **name similarity** (5), **address similarity** (4), **numeric/structural** (3), and **binary flags** (3).

In [ ]:
FEATURE_DESCRIPTIONS = {
    'name_token_jaccard':       'Jaccard similarity of word-token sets in preprocessed names',
    'name_jaro_winkler':        'Jaro-Winkler similarity; rewards shared prefixes',
    'name_levenshtein_ratio':   'Edit-distance ratio (rapidfuzz); handles typos',
    'name_char_trigram_cosine': 'Cosine similarity of character-trigram TF vectors',
    'name_len_diff_ratio':      'Absolute token-count difference / max length',
    'addr_token_jaccard':       'Jaccard similarity of word-token sets in addresses',
    'addr_jaro_winkler':        'Jaro-Winkler on full address strings',
    'addr_levenshtein_ratio':   'Edit-distance ratio on address strings',
    'addr_char_trigram_cosine': 'Cosine of char-trigram vectors for addresses',
    'numeric_overlap_jaccard':  'Jaccard on digit tokens (house / zip numbers)',
    'zip_code_match':           'Binary: 1 if 5-digit ZIP codes match exactly',
    'is_script_mismatch':       'Binary: 1 if one name uses non-Latin script',
    'both_addresses_missing':   'Binary: 1 if both records have no address',
    'one_address_missing':      'Binary: 1 if exactly one record has no address',
    'same_country':             'Binary: 1 if s1_country == probe_country',
}

feat_table = pd.DataFrame([
    {'#': i + 1, 'Feature': f, 'Description': FEATURE_DESCRIPTIONS[f]}
    for i, f in enumerate(FEATURE_COLUMNS)
])
feat_table.style.set_properties(**{'text-align': 'left'}).hide(axis='index')

### 3. Feature Statistics

Descriptive statistics reveal the range, spread, and skewness of each feature. Binary features (zip_code_match, same_country, etc.) should have mean = prevalence rate. Continuous similarity features should ideally be well-spread across [0, 1].

In [ ]:
stats = feat_df[FEATURE_COLUMNS].describe().round(3).T
stats.style.background_gradient(cmap='YlOrRd', subset=['mean', 'std'])

### 4. Feature Means: Match vs Non-Match

The most diagnostic check: for a feature to be useful, its mean should **differ** substantially between matched pairs (label=1) and non-matched pairs (label=0).

In [ ]:
group_means = feat_df.groupby('label')[FEATURE_COLUMNS].mean().T
group_means.columns = {0: 'Non-Match (label=0)', 1: 'Match (label=1)'}\
    .get(0, 'Non-Match'), {0: 'Non-Match (label=0)', 1: 'Match (label=1)'}.get(1, 'Match')

# Safer rename
group_means = feat_df.groupby('label')[FEATURE_COLUMNS].mean().T
rename_map = {c: ('Non-Match (label=0)' if c == 0 else 'Match (label=1)') for c in group_means.columns}
group_means = group_means.rename(columns=rename_map)
group_means['Delta'] = (group_means['Match (label=1)'] - group_means['Non-Match (label=0)']).round(4)

group_means.round(4).style\
    .background_gradient(cmap='RdYlGn', subset=['Delta'])\
    .format(precision=4)

### 5. Correlation Heatmap

High inter-feature correlation indicates redundancy. The model can still use correlated features (tree ensembles handle this well), but understanding clusters helps feature selection and model interpretability.

In [ ]:
corr = feat_df[FEATURE_COLUMNS].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix', fontsize=15, fontweight='bold', pad=12)
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('results/plots/feature_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 6. Feature Distributions by Label

KDE plots show the full distribution of the three most discriminative name features, split by match vs non-match. Good features show minimal overlap between distributions.

In [ ]:
key_feats = ['name_token_jaccard', 'name_jaro_winkler', 'name_levenshtein_ratio']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, feat in zip(axes, key_feats):
    for lbl, lname, color in [(1, 'Match', '#2980b9'), (0, 'Non-Match', '#e74c3c')]:
        subset = feat_df[feat_df['label'] == lbl][feat].dropna()
        sns.kdeplot(subset, ax=ax, label=lname, color=color, fill=True, alpha=0.35)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Value')
    ax.legend()

plt.suptitle('Feature Distributions: Match vs Non-Match', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/feature_distributions_kde.png', dpi=150, bbox_inches='tight')
plt.show()